# 15.07 - CV Error Analysis

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** A reusable `wrong_prediction_grid()` utility that displays each image with its true label, predicted label, and confidence.

Accuracy and Macro-F1 tell us *how much* a model is wrong. Error analysis helps us understand *why*. Today you will turn validation predictions into a compact report, rank the most confident mistakes, inspect class confusions, and compare performance across blur, lighting, and pose slices.


## Core Ideas

### 1. Start with validation predictions

Error analysis must use data that was not used to update model weights. For every validation example, keep the image index, true label, predicted label, class probabilities, confidence, and whether the prediction is correct.

### 2. Confidence is useful but not certainty

For a softmax classifier, confidence is the largest predicted probability. A high-confidence wrong prediction is especially informative: it often reveals a systematic shortcut, a mislabeled sample, or a missing training pattern. Softmax scores need not be calibrated, so treat them as ranking signals rather than guaranteed probabilities of correctness.

### 3. Confusion matrices reveal class-level structure

Rows represent true classes and columns represent predicted classes. Large off-diagonal counts expose recurring class pairs. Always preserve a documented label-to-index mapping so the matrix is interpretable.

### 4. Slice analysis tests hypotheses

Overall metrics can hide weak subgroups. Compare accuracy for slices such as `sharp` versus `blurred`, `bright` versus `dark`, and `front` versus `tilted`. A slice with few samples is a clue, not a final conclusion; record its sample count beside its metric.

### 5. A disciplined loop

Measure, inspect representative failures, form one hypothesis, make one targeted change, and re-measure on the same validation protocol. Avoid changing augmentation, architecture, loss, and data cleaning simultaneously because the source of improvement becomes unclear.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ["circle", "square", "triangle"]


## Prepared Validation Data

The cell below provides a deterministic mini validation set. Each image has a class-colored shape-like patch plus controlled brightness and blur metadata. The logits deliberately include several confident mistakes so the diagnostic utilities have meaningful cases to inspect. Data preparation is complete and is not an exercise.


In [ ]:
def make_day15_validation_data(n_samples=18, image_size=24):
    images = torch.zeros(n_samples, 3, image_size, image_size, dtype=torch.float32)
    labels = torch.tensor([i % 3 for i in range(n_samples)], dtype=torch.long)
    blur_slice = []
    lighting_slice = []
    pose_slice = []

    yy, xx = np.mgrid[:image_size, :image_size]
    for i in range(n_samples):
        label = int(labels[i])
        brightness = 0.55 if i % 2 == 0 else 0.85
        image = torch.full((3, image_size, image_size), 0.08 * brightness)
        center_x = image_size // 2 + (-2 if i % 3 == 0 else 2 if i % 3 == 1 else 0)
        center_y = image_size // 2
        mask = (xx - center_x) ** 2 + (yy - center_y) ** 2 <= 36
        image[label, torch.from_numpy(mask)] = brightness
        images[i] = image
        blur_slice.append("blurred" if i in [2, 5, 8, 11, 14, 17] else "sharp")
        lighting_slice.append("dark" if i % 2 == 0 else "bright")
        pose_slice.append("tilted" if i % 3 == 0 else "front")

    logits = torch.full((n_samples, 3), -0.8, dtype=torch.float32)
    logits[torch.arange(n_samples), labels] = 2.2
    forced_errors = {2: 1, 5: 0, 8: 1, 11: 0, 14: 0}
    for index, wrong_class in forced_errors.items():
        logits[index] = -1.0
        logits[index, wrong_class] = 3.4
        logits[index, labels[index]] = 0.2

    slices = {
        "blur": blur_slice,
        "lighting": lighting_slice,
        "pose": pose_slice,
    }
    return images, labels, logits, slices


val_images, val_labels, val_logits, val_slices = make_day15_validation_data()
print("images:", tuple(val_images.shape), val_images.dtype)
print("labels:", tuple(val_labels.shape), val_labels.dtype)
print("device:", val_logits.device)
print("label mapping:", dict(enumerate(CLASS_NAMES)))


## Exercise 15-A: Build Prediction Records

Implement `build_prediction_records(logits, true_labels)`. Apply softmax across classes and return a list of dictionaries. Each record must contain `index`, `true_index`, `pred_index`, `confidence`, `probabilities`, and `correct`. Store indices as Python integers, confidence as a Python float, probabilities as a detached CPU tensor, and `correct` as a Python Boolean.


In [ ]:
def build_prediction_records(logits, true_labels):
    if logits.ndim != 2:
        raise ValueError("logits must have shape [N, C]")
    if true_labels.ndim != 1 or len(true_labels) != len(logits):
        raise ValueError("true_labels must have shape [N]")
    probabilities = torch.softmax(logits.detach(), dim=1).cpu()
    confidences, predictions = probabilities.max(dim=1)
    labels_cpu = true_labels.detach().cpu()
    records = []
    for index in range(len(labels_cpu)):
        true_index = int(labels_cpu[index])
        pred_index = int(predictions[index])
        records.append({
            "index": index,
            "true_index": true_index,
            "pred_index": pred_index,
            "confidence": float(confidences[index]),
            "probabilities": probabilities[index].clone(),
            "correct": bool(pred_index == true_index),
        })
    return records


prediction_records = build_prediction_records(val_logits, val_labels)
print("validation mistakes:", sum(not record["correct"] for record in prediction_records))


## Exercise 15-B: Rank Wrong Predictions

Implement `select_wrong_predictions(records, max_items=6)`. Keep only incorrect records, order them by confidence from highest to lowest, and return at most `max_items`. This makes the most confident mistakes the first candidates for visual inspection.


In [ ]:
def select_wrong_predictions(records, max_items=6):
    if max_items < 0:
        raise ValueError("max_items must be non-negative")
    wrong = [record for record in records if not record["correct"]]
    wrong.sort(key=lambda record: record["confidence"], reverse=True)
    return wrong[:max_items]


ranked_mistakes = select_wrong_predictions(prediction_records)
print([(record["index"], round(record["confidence"], 3)) for record in ranked_mistakes])


## Exercise 15-C: Create the Wrong-Prediction Grid

Implement `wrong_prediction_grid(images, records, class_names, max_items=6, columns=3)`. Use the ranked mistakes from Exercise 15-B. Display CHW tensors as HWC images, clip display values to `[0, 1]`, and title each panel with the true label, predicted label, and confidence. Hide unused axes and return `(figure, axes)` so the utility can be tested and reused.


In [ ]:
def wrong_prediction_grid(images, records, class_names, max_items=6, columns=3):
    if images.ndim != 4 or images.shape[1] not in [1, 3]:
        raise ValueError("images must have shape [N, C, H, W] with C equal to 1 or 3")
    if columns <= 0:
        raise ValueError("columns must be positive")
    selected = select_wrong_predictions(records, max_items=max_items)
    panel_count = max(1, len(selected))
    rows = int(np.ceil(panel_count / columns))
    figure, axes_grid = plt.subplots(rows, columns, figsize=(4 * columns, 3.5 * rows))
    axes = np.atleast_1d(axes_grid).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, record in zip(axes, selected):
        image = images[record["index"]].detach().cpu().permute(1, 2, 0).numpy()
        if image.shape[2] == 1:
            axis.imshow(np.clip(image[:, :, 0], 0.0, 1.0), cmap="gray")
        else:
            axis.imshow(np.clip(image, 0.0, 1.0))
        true_name = class_names[record["true_index"]]
        pred_name = class_names[record["pred_index"]]
        axis.set_title(f"true: {true_name}\npred: {pred_name} | conf: {record['confidence']:.1%}")
        axis.axis("off")
    if not selected:
        axes[0].set_title("No wrong predictions")
    figure.tight_layout()
    return figure, axes


error_figure, error_axes = wrong_prediction_grid(
    val_images, prediction_records, CLASS_NAMES, max_items=6, columns=3
)
plt.close(error_figure)


## Exercise 15-D: Count and Rank Class Confusions

Implement `confusion_counts(records, num_classes)` so rows are true classes and columns are predictions. Then implement `top_confusions(matrix, class_names, top_k=3)` to return the largest non-diagonal confusions as dictionaries with `true_label`, `pred_label`, and `count`. Exclude zero-count pairs.


In [ ]:
def confusion_counts(records, num_classes):
    matrix = torch.zeros((num_classes, num_classes), dtype=torch.long)
    for record in records:
        matrix[record["true_index"], record["pred_index"]] += 1
    return matrix


def top_confusions(matrix, class_names, top_k=3):
    pairs = []
    for true_index in range(matrix.shape[0]):
        for pred_index in range(matrix.shape[1]):
            count = int(matrix[true_index, pred_index])
            if true_index != pred_index and count > 0:
                pairs.append({
                    "true_label": class_names[true_index],
                    "pred_label": class_names[pred_index],
                    "count": count,
                })
    pairs.sort(key=lambda item: (-item["count"], item["true_label"], item["pred_label"]))
    return pairs[:top_k]


confusion_matrix = confusion_counts(prediction_records, len(CLASS_NAMES))
print(confusion_matrix)
print(top_confusions(confusion_matrix, CLASS_NAMES))


## Exercise 15-E: Measure Slice Accuracy

Implement `slice_accuracy(records, slice_values)`. Return one dictionary per unique slice value containing `count`, `correct`, and `accuracy`. Use prediction-record order, validate that the lengths match, and sort slice names for stable reports. Apply it to blur, lighting, and pose metadata before deciding which data or augmentation change to try next.


In [ ]:
def slice_accuracy(records, slice_values):
    if len(records) != len(slice_values):
        raise ValueError("records and slice_values must have the same length")
    report = {}
    for slice_name in sorted(set(slice_values)):
        matching = [
            record for record, value in zip(records, slice_values)
            if value == slice_name
        ]
        correct = sum(record["correct"] for record in matching)
        report[slice_name] = {
            "count": len(matching),
            "correct": int(correct),
            "accuracy": correct / len(matching),
        }
    return report


slice_report = {
    slice_type: slice_accuracy(prediction_records, values)
    for slice_type, values in val_slices.items()
}
for slice_type, report in slice_report.items():
    print(slice_type, report)


## Test Cases

Run this cell after completing all TODO cells. The tests check probability shape and normalization, ranking, image-grid titles, confusion counts, slice accounting, and the true/predicted label mapping. A correct implementation prints `Day 15 tests passed`.


In [ ]:
def run_day15_tests():
    records = build_prediction_records(val_logits, val_labels)
    assert len(records) == len(val_labels)
    assert records[0]["probabilities"].shape == (len(CLASS_NAMES),)
    assert records[0]["probabilities"].dtype == torch.float32
    assert records[0]["probabilities"].device.type == "cpu"
    assert torch.isclose(records[0]["probabilities"].sum(), torch.tensor(1.0), atol=1e-6)
    assert isinstance(records[0]["correct"], bool)

    wrong = select_wrong_predictions(records, max_items=3)
    assert len(wrong) == 3
    assert all(not record["correct"] for record in wrong)
    assert wrong[0]["confidence"] >= wrong[1]["confidence"] >= wrong[2]["confidence"]

    figure, axes = wrong_prediction_grid(val_images, records, CLASS_NAMES, max_items=4, columns=2)
    titled_axes = [axis for axis in axes if axis.get_title()]
    assert len(titled_axes) == 4
    assert all("true:" in axis.get_title() and "pred:" in axis.get_title() for axis in titled_axes)
    plt.close(figure)

    matrix = confusion_counts(records, len(CLASS_NAMES))
    assert matrix.shape == (3, 3)
    assert matrix.dtype == torch.long
    assert int(matrix.sum()) == len(records)
    assert int(matrix.sum() - matrix.diag().sum()) == len(select_wrong_predictions(records, 99))
    confusions = top_confusions(matrix, CLASS_NAMES, top_k=2)
    assert len(confusions) <= 2
    assert all(item["true_label"] != item["pred_label"] for item in confusions)

    blur_report = slice_accuracy(records, val_slices["blur"])
    assert set(blur_report) == {"blurred", "sharp"}
    assert sum(item["count"] for item in blur_report.values()) == len(records)
    assert all(0.0 <= item["accuracy"] <= 1.0 for item in blur_report.values())
    print("Day 15 tests passed")


run_day15_tests()


## Day 15 Checklist

- [ ] I can explain why high-confidence mistakes are useful but do not prove certainty.
- [ ] I can preserve the label mapping from logits to class names.
- [ ] I can create and read a true-row/predicted-column confusion matrix.
- [ ] My wrong-prediction grid shows image, true label, predicted label, and confidence.
- [ ] I report both sample count and accuracy for each blur, lighting, or pose slice.
- [ ] I can propose one targeted next experiment based on the strongest error pattern.
